In [2]:
!uv add langchain

Resolved 237 packages in 2ms
Checked 231 packages in 7ms


In [113]:
!uv add tiktoken

Resolved 237 packages in 2ms
Checked 231 packages in 207ms


In [88]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [89]:
from langchain.agents import create_agent

In [90]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage

In [91]:
from langchain.tools import tool

In [92]:
from typing import Union, List, Optional, Dict

In [93]:
import tiktoken

In [94]:
from dotenv import load_dotenv

In [95]:
import numpy as np

In [96]:
from pydantic import BaseModel, Field

In [97]:
import os

In [98]:
load_dotenv()

True

In [99]:
llm = ChatOpenAI(
    base_url = os.getenv("BASE_URL"),
    api_key = os.getenv("API_KEY"),
    model = os.getenv("LLM"),
)

In [100]:
emb = OpenAIEmbeddings(
    base_url= os.getenv("BASE_URL"),
    api_key=os.getenv("API_KEY"),
    model=os.getenv("EMB"),
)

In [101]:
@tool("calculator", description="Performs arithmetic calculations. Use this for any math problems.")
def calculator(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))

In [102]:
messages = [
    SystemMessage("You are a helpful mathematical assistant"),
    HumanMessage("Solve this query using a given tool : 11-1")
]

In [121]:
agent = create_agent(
    model = llm,
    tools = [calculator],
)

In [122]:
response = agent.invoke({
    "messages" : messages
})

In [123]:
response

{'messages': [SystemMessage(content='You are a helpful mathematical assistant', additional_kwargs={}, response_metadata={}, id='82ce7f1b-9ed0-4cd2-aa1e-7aa4118f9b62'),
  HumanMessage(content='Solve this query using a given tool : 11-1', additional_kwargs={}, response_metadata={}, id='722069db-a8a4-4737-9f3e-76fc61366248'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 295, 'total_tokens': 365, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Deepseek-vapt', 'system_fingerprint': 'vllm-0.25.0-tp2-ep-d4f8ac0c', 'id': 'chatcmpl-90fe04a5db7b3465', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a09f92-95bb-7c83-b334-4339921f208e-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '11-1'}, 'id': 'chatcmpl-tool-bc9b8d4cb0323536', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 295, 'out

In [124]:
response['messages'][-1].content

'The result of \\(11 - 1\\) is **10**.'

In [125]:
class User(BaseModel):
    name: str = Field(..., description= "Name of the user")
    age: int = Field(..., description= "Age of the user")

class UserDetails(BaseModel):
    details: List[User] = Field(..., description= "List of user details")

In [141]:
class Agent:
    def __init__(self, model: ChatOpenAI, tools:List):
        self.model = model
        self.tools = tools
        if self.tools:
            self.model = self.model.bind_tools(self.tools)
        self.tool_map = {tool.name: tool for tool in self.tools}

    def parse(self, schema, message: str):
        parser = self.model.with_structured_output(schema, strict=True)
        return parser.invoke(message)

    def invoke(self, thread: Thread):
        while True:
            response = self.model.invoke(thread.messages)
            thread.append(response)

            if response.tool_calls:
                for tool in response.tool_calls:
                    arg = tool["arg"]
                    call_id = tool["id"]
                    name = tool["name"]
                    result = self.tool_map[name].invoke(arg)
                    thread.append(ToolMessage(name=name, content=result, tool_call_id=call_id))
            else:
                return response
                
    def __ror__(self, thread: Thread):
        agent.invoke(thread)

In [142]:
class Thread:
    def __init__(
        self, 
        messages: List[Union[SystemMessage, HumanMessage, AIMessage, ToolMessage]] = None,
        system_prompt: Union[str, SystemMessage] = None,
        compression_prompt: str = None,
        token_limit: int = None,
        thread_hide_rules: List[ThreadHideRule] = None
    ):
        self.messages = []
        self.system_prompt = system_prompt
        self.compression_prompt = compression_prompt
        self.token_limit = token_limit
        self.encoder = tiktoken.encoding_for_model('gpt-4o-mini')
        self.agent = None
        self.root = None
        self.parent = None
        self.child = None
        self.tail = None
        self.thread_hide_rules = thread_hide_rules

        if messages is not None:
            index = self._find_index(messages)
            if index == -1 or index == 0:
                self.messages = messages
            else:
                raise ValueError(
                    f"system msg is not at the starting index and found at index {index} index"
                )
         if self.system_prompt is not None:
            index = self._find_index(self.messages)
            if isinstance(self.system_prompt, str):
                
            
                
            
            
            
            

        
        
    def _find_index(self, messages: List[Union[HumanMessage,AIMessage, SystemMessage, ToolMessage]]):
        for i in range(len(messages)):
            if isinstance(messages[i], SystemMessage):
                return i
        return -1

IndentationError: unindent does not match any outer indentation level (<string>, line 30)

In [143]:
message = "This is the codebase for agent class, vector class and thread"
document = "Building upon the foundational models of the Qwen3 series, Qwen3 Embedding provides a comprehensive range of text embeddings models in various sizes"

In [144]:
class Vector:
    def __init__(self, model: OpenAIEmbeddings, dim: Union[int, bool] = False):
        self.model = model
        self.dim = dim

    def _normalize_dim(self, arr: np.ndarray):
        arr = arr[:self.dim]
        norms = np.linalg.norm(arr, keepdims=True)
        norms[norms == 0] = 1
        return arr / norms
        

    def embed(self, document: Union[str, List[str]]):
        if isinstance(document, str):
            embed_query = np.array(self.model.embed_query(document))
            if self.dim and self.dim <= len(embed_query):
                return self._normalize_dim(embed_query)
            return embed_query
            
        elif isinstance(document, list):
            embed_document = np.array(self.model.embed_documents(document))
            if self.dim and self.dim <= len(embed_document[0]):
                arr = []
                for emb in embed_document:
                    arr.append(self._normalize_dim(emb))
                return np.array(arr)
            return embed_document

In [145]:
vec = Vector(model = emb)

In [146]:
e = vec.embed([message, document])

In [147]:
e0 = e[0]
e1 = e[1]

In [148]:
e0@e1

np.float64(0.47347420937861984)

In [149]:
e0

array([-0.00658808,  0.02931992, -0.01044596, ..., -0.00824994,
       -0.00379853, -0.00641002], shape=(4096,))

In [150]:
e1

array([ 0.00193897, -0.0029819 ,  0.00376043, ..., -0.013044  ,
       -0.00840221, -0.00376043], shape=(4096,))

In [163]:
message = f"""
These are the user details:
harry : 20, jan : 30, feb : 40, mar : 50
return the user details in structured json schema

{UserDetails.model_json_schema()}
"""

In [164]:
myagent= Agent(
    model = llm,
    tools = [calculator]
)

In [165]:
response = myagent.parse(schema = UserDetails, message = message)

In [166]:
response

UserDetails(details=[User(name='harry', age=20), User(name='jan', age=30), User(name='feb', age=40), User(name='mar', age=50)])

In [167]:
response.details

[User(name='harry', age=20),
 User(name='jan', age=30),
 User(name='feb', age=40),
 User(name='mar', age=50)]